# Tutorial 5: Hybrid Coherent FDMT (CFDMT)

In this tutorial, we study the physical limits of incoherent dedispersion and learn how the **Hybrid Coherent FDMT (CFDMT)** algorithm achieves microsecond time resolution from raw voltage baseband data.


## 1. The Coherent Dedispersion Limit

In Tutorials 1–4, we performed **incoherent dedispersion**: summing total power (detected intensity) across frequency channels.

However, recall the intra-channel smearing formula:

$$\Delta t_{\text{smear}} \approx 2 \mathcal{D} \cdot \text{DM} \cdot \nu^{-3} \Delta \nu$$

Once power is detected, intra-channel smearing is **irreversible**. Even if your sampling clock is $1\,\mu\text{s}$, at $1.4\,\text{GHz}$ with a $1\,\text{MHz}$ channel, a $\text{DM}=100\,\text{pc}\,\text{cm}^{-3}$ pulse will be smeared across:

$$\Delta t_{\text{smear}} \approx 2 (4148.8) (100) (1400)^{-3} (1.0) \approx 302\,\mu\text{s}$$

Any sub-millisecond structure (such as pulsar micro-structure, magnetar bursts, or FRB sub-bursts) is irrevocably blurred!

### Coherent Dedispersion: Inverting the Chirp in Voltage
To achieve the true physical time resolution $\sim 1/B$, we must operate directly on the complex electric field voltages $V(t) = V_x(t) + i V_y(t)$ before detection.
The interstellar plasma acts as a linear filter with transfer function:

$$H(\nu) = \exp\left( -i \frac{2\pi \mathcal{D} \cdot \text{DM}}{\nu_0^2 (\nu_0 + f)} f^2 \right)$$

By multiplying the Fourier-transformed baseband voltages $\tilde{V}(f)$ by the inverse chirp $H^{-1}(f)$, dispersion is completely undone down to the Nyquist limit!

---

## 2. The Hybrid Coherent FDMT (CFDMT) Architecture

Coherent dedispersion at thousands of fine DM trials requires immense FFT compute power.
The **CFDMT** hybrid algorithm bridges this gap:
1. Divide the baseband voltage stream into $N_{\text{sub}}$ subbands.
2. Apply **coarse coherent dechirping** at widely spaced DM steps $\text{DM}_{\text{coarse}}$.
3. Detect total intensity (Stokes I) at the required detected sampling interval $t_p$.
4. Run a fast, fine **incoherent FDMT** around each coarse DM.

This delivers the **full sensitivity of coherent dedispersion** at a fraction of the computational cost!


In [1]:
from dmtlib import CohFDMTPlan

# Coherent FDMT configuration. nbin must be larger than twice the
# overlap-save length (default 8192 samples).
f_center = 1400.0   # Center frequency in MHz
sub_bw = 16.0       # Bandwidth per subband in MHz
nsub = 8            # 8 subbands -> total bandwidth = 128 MHz
tbin = 1.0 / (sub_bw * 1e6)  # Nyquist baseband voltage sampling interval
nbin = 32768        # FFT size per coherent block
nfft = 2            # Number of FFT blocks per segment
tp = 64e-6          # Output detected power time resolution (64 microseconds)

dm_min = 50.0       # pc cm^-3
dm_max = 60.0       # Coherent search range

cfdmt_plan = CohFDMTPlan(
    fcenter=f_center,
    bwsub=sub_bw,
    nsub=nsub,
    tbin=tbin,
    nbin=nbin,
    nfft=nfft,
    t_p=tp,
    dm_max=dm_max,
    dm_min=dm_min,
)

print("CohFDMTPlan summary:")
print(f"  Total bandwidth: {cfdmt_plan.bw:.1f} MHz")
print(f"  Overlap:         {cfdmt_plan.noverlap} samples")
print(f"  Coarse DM trials: {len(cfdmt_plan.dm_grid_coh)}")
print(f"  Final DM trials:  {cfdmt_plan.ndm}")
print(f"  Fine delay trials: {cfdmt_plan.dt_max}")


CohFDMTPlan summary:
  Total bandwidth: 128.0 MHz
  Overlap:         8192 samples
  Coarse DM trials: 1
  Final DM trials:  2048
  Fine delay trials: 1023


## 3. Baseband Unpacking and Layouts

Raw baseband voltages are recorded by digital receivers in packed integer formats (e.g. 8-bit or 4-bit real/imaginary values across polarizations).

DMT natively supports standard radio astronomy baseband data orderings:
- `'PRITF'`: Polarization, Real/Imaginary, Time, Frequency
- `'FTPRI'`: Frequency, Time, Polarization, Real/Imaginary
- `'RITFP'`: Real/Imaginary, Time, Frequency, Polarization

The C++ unpacker SIMD engine seamlessly unpacks, de-interleaves, and feeds voltage streams into FFTW / cuFFT execution pipelines.

```python
# Initialize CohFDMTCPU engine
cfdmt = CohFDMTCPU(
    f_center=f_center,
    sub_bw=sub_bw,
    nsub=nsub,
    tbin=tbin,
    nbin=nbin,
    nfft=nfft,
    tp=tp,
    dm_max=dm_max,
    dm_min=dm_min,
    data_order="PRITF",
)
# Resulting dmt_plane = cfdmt.execute(voltage_buffer)
```


## Summary

- Incoherent dedispersion is fundamentally limited by intra-channel dispersion smearing $\Delta t_{\text{smear}}$.
- Baseband coherent dedispersion restores full temporal resolution by multiplying voltage spectra with the inverse phase response $H^{-1}(f)$.
- Hybrid Coherent FDMT (`CohFDMTCPU` / `CohFDMTPlan`) combines coarse coherent de-chirping with fine incoherent FDMT, unlocking high-resolution FRB and pulsar searches.

In **Tutorial 6**, we explore **Real-Time Streaming Pipelines and Multi-Beam Batching**!
